# DATA

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None) # To show all columns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")

print('Shape of training data: ', train.shape)
print('Shape of testing data: ', test.shape)


# EDA AND PRE PROCESSING

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

#### Key Insights:
- emoticon, upvot, downvote and if columns have low mean and very high max values they seem very right skewed.
- The label has more than half it's values as 0, seems very imbalanced.

In [ ]:
train.dtypes

In [ ]:
test.dtypes

### Duplicate value check

In [ ]:
train[train.duplicated()]

In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
temp_train = train[train['race'].isna()]
print(temp_train.shape)

temp_test = test[test['race'].isna()]
print(temp_test.shape)

In [ ]:
temp_train.isna().sum()

In [ ]:
temp_test.isna().sum()

#### Key Insights:
- All the null values in race, religion and gender are together for both train and test.

In [ ]:
label_distribution = train['label'].value_counts(normalize=True)
print(label_distribution)
labs = label_distribution.index
proportion = label_distribution.values
plt.pie(x=proportion,labels=labs,autopct='%1.1f%%')
plt.show()

#### Key Insights:
- The label distribution is very imbalanced.
- Class 3 and 4 account for <11%.

In [ ]:
train.nunique()

In [ ]:
cat_cols = ['race','religion','gender']
for col in cat_cols:
    print(col,':\n',train[col].unique(),'\n',test[col].unique(),'\n')

All the categorical values in train are present in test

In [ ]:
for col in cat_cols:
    distribution = train[col].value_counts(normalize=True)
    
    # Sirf top 4 labels show karne ke liye
    top4 = distribution.nlargest(4)
    rest = distribution.iloc[4:].sum()
    final_dist = top4.copy()
    final_dist['rest'] = rest
    
    print(distribution)
    labs = final_dist.index
    proportion = final_dist.values
    
    plt.pie(x=proportion, labels=labs)
    plt.show()

#### Key insights:
- The categoricals table are dominated by none label.(Excluding the null values)
- Many categories have ~1 of <1 probablities.
- This shows imbalanced distribution of categorical values.

In [ ]:
corr_data = train.corr(numeric_only=True)
print(corr_data)
sns.heatmap(corr_data, cmap='coolwarm')

#### Key Insights:
- if_2 seems to have the highest correlation with label.
- upvote and downvote have high correlation they likely won't help individually but could be used in some form.
- Most features have very low correlation. Don't seem linear.

In [ ]:
postids = train['post_id'].unique()
test_comon = test[test['post_id'].isin(postids)]
test_comon.shape

#### Key insight:
- There are significant postids in train that are also in test.

# FEATURE ENGINEERING 

In [ ]:
train['created_date'] = pd.to_datetime(train['created_date'])
test['created_date'] = pd.to_datetime(test['created_date'])

In [ ]:
train['race'] = train['race'].fillna('unknown')
train['religion'] = train['religion'].fillna('unknown')
train['gender'] = train['gender'].fillna('unknown')

test['race'] = test['race'].fillna('unknown')
test['religion'] = test['religion'].fillna('unknown')
test['gender'] = test['gender'].fillna('unknown')

In [ ]:
string_cols = ['race','religion','gender','comment']
for i in string_cols:
    train[i] = train[i].astype('string')
    test[i] = test[i].astype('string')

In [ ]:
train['vote_score'] = train['upvote'] - train['downvote']
train['vote_total'] = train['upvote'] + train['downvote']

test['vote_score'] = test['upvote'] - test['downvote']
test['vote_total'] = test['upvote'] + test['downvote']


train['year'] = train['created_date'].dt.year
train['month']= train['created_date'].dt.month
train['day'] = train['created_date'].dt.day
train['hour'] = train['created_date'].dt.hour
train['week_day'] = train['created_date'].dt.dayofweek
train['is_weekend'] = train['week_day'] >= 5
train['is_weekend'] = train['is_weekend'].astype(int)

test['year'] = test['created_date'].dt.year
test['month']= test['created_date'].dt.month
test['day'] = test['created_date'].dt.day
test['hour'] = test['created_date'].dt.hour
test['week_day'] = test['created_date'].dt.dayofweek
test['is_weekend'] = test['week_day'] >= 5
test['is_weekend'] = test['is_weekend'].astype(int)

train['total_emoticons'] = train['emoticon_1'] + train['emoticon_2'] + train['emoticon_3']
test['total_emoticons'] = test['emoticon_1'] + test['emoticon_2'] + test['emoticon_3']

train['comment'] = train['comment'].fillna('')
test['comment'] = test['comment'].fillna('')

train['comment_length'] = train['comment'].str.len()
train['word_count'] = train['comment'].str.split().str.len()

test['comment_length'] = test['comment'].str.len()
test['word_count'] = test['comment'].str.split().str.len()

train['comment_length'] = train['comment_length'].astype('int64')
train['word_count'] = train['word_count'].astype('int64')

test['comment_length'] = test['comment_length'].fillna(0).astype('int64')
test['word_count'] = test['word_count'].astype('int64')

train['disability'] = train['disability'].astype('int32')
test['disability'] = test['disability'].astype('int32')

In [ ]:
train['has_question'] = train['comment'].str.contains(r'\?').astype(int)
test['has_question'] = test['comment'].str.contains(r'\?').astype(int)

train['has_exclamation'] = train['comment'].str.contains('!').astype(int)
test['has_exclamation'] = test['comment'].str.contains('!').astype(int)

In [ ]:
train = train.drop(columns=['created_date'])
test = test.drop(columns=['created_date'])

In [ ]:
train.dtypes

### OUTLIER CHECK

In [ ]:
num_cols = ['upvote', 'downvote', 'if_1', 'if_2','vote_score', 'vote_total','comment_length', 'word_count', 'total_emoticons']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for i, col in enumerate(num_cols):
    row = i // 3
    col_idx = i % 3
    
    axes[row, col_idx].boxplot(train[col])
    axes[row, col_idx].set_title(col)

plt.tight_layout()
plt.show()

#### Key Insights:
- Data has a lot of outliers, but they could signals so they are not removed.
- Most columns have a flat box, and the contrast between them and the outliers could be a signal.
- Most columns have outliers in 1 direction, this can be handled in skewness.

### SKEWNESS HANDLING

In [ ]:
skew_vals = train[num_cols].skew().sort_values(ascending=False)
print(skew_vals)

feature_names = skew_vals.index
skew_scores = skew_vals.values

plt.figure(figsize=(10,6))
plt.barh(feature_names,skew_scores)
plt.xlabel('Skewness')
plt.title('Skewness of numeric features')
plt.show()

In [ ]:
num_cols = ['upvote', 'downvote', 'if_1', 'if_2','vote_score', 'vote_total','comment_length', 'word_count', 'total_emoticons']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for i, col in enumerate(num_cols):
    row = i // 3
    col_idx = i % 3
    
    axes[row, col_idx].hist(train[col], bins=500)
    axes[row, col_idx].set_title(col)
    print(col,':  min',min(train[col]),'max: ',max(train[col]))

plt.tight_layout()
plt.show()

#### Key Insights:
- The datset seems highly skewed.
- if_1 and if_2 have huge skews, they same to have rare high value event. This could have latent signals.
- Upvote, downvot and vote_score having a high skew mean that the engagements in most posts is low but some have very high engagement. These must be viral posts.
- Word length and comment length have some skew but don't necessarily require handling.

In [ ]:
log_cols = ['upvote', 'downvote', 'if_1', 'if_2', 'vote_total', 'total_emoticons']
for col in log_cols:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

train['vote_score'] = np.sign(train['vote_score']) * np.log1p(np.abs(train['vote_score']))
test['vote_score'] = np.sign(test['vote_score']) * np.log1p(np.abs(test['vote_score']))

#### Only required linear models models. But tree models can also use as helps in saving time in handling extreme splits.

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train_backup = train.copy()
test_backup = test.copy()

# MODELS

In [ ]:
model_performance = {}

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

y = train['label']
x = train.drop('label',axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    x,y,
    train_size=0.8,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

### One Hot Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown = 'ignore')

X_train_cat = ohe.fit_transform(X_train[['race','religion','gender']])
X_val_cat = ohe.transform(X_val[['race','religion','gender']])

X_train = X_train.drop(columns=['race','religion','gender'])
X_val = X_val.drop(columns=['race','religion','gender'])

### Tf- Idf Vectorisation

In [ ]:
X_train['comment'] = X_train['comment'].str.lower()
X_val['comment'] = X_val['comment'].str.lower()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_vectorizer = TfidfVectorizer(
    stop_words='english',
    min_df=5,
    max_df=0.9,
    max_features=50000,
    ngram_range=(1,2)
)

char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    min_df=3,
    max_features=40000
)

In [ ]:
X_train_word = word_vectorizer.fit_transform(X_train['comment'])
X_val_word = word_vectorizer.transform(X_val['comment'])

X_train_char = char_vectorizer.fit_transform(X_train['comment'])
X_val_char = char_vectorizer.transform(X_val['comment'])

In [ ]:
from scipy.sparse import hstack

X_train_text = hstack([X_train_word, X_train_char])
X_val_text = hstack([X_val_word, X_val_char])

## Feature Scaling

In [ ]:
X_train['disability'] = X_train['disability'].astype(int)
X_val['disability'] = X_val['disability'].astype(int)

X_train['is_weekend'] = X_train['is_weekend'].astype(int)
X_val['is_weekend'] = X_val['is_weekend'].astype(int)

num_cols = ['post_id','emoticon_1','emoticon_2','emoticon_3','upvote','downvote','if_1'
            ,'if_2','disability','vote_score','vote_total','month','year','day','week_day',
            'is_weekend','total_emoticons','comment_length','word_count']
            
X_train_num = X_train[num_cols].values
X_val_num = X_val[num_cols].values

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train_num)
X_val_scaled = scaler.transform(X_val_num)

## Feature Combining

In [ ]:
from scipy.sparse import hstack

X_train_final = hstack([X_train_text, X_train_cat, X_train_scaled])
X_val_final = hstack([X_val_text, X_val_cat, X_val_scaled])

# LOGISTIC REGRESSION BASELINE

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1
)

model.fit(X_train_final, y_train)
y_preds = model.predict(X_val_text)

In [ ]:
print(classification_report(y_val, y_preds))

              precision    recall  f1-score   support

           0       0.96      0.93      0.95     22835
           1       0.60      0.72      0.65      3183
           2       0.86      0.58      0.69     12488
           3       0.16      0.77      0.26      1094

    accuracy                           0.80     39600
    macro avg      0.64      0.75      0.64     39600
    weighted avg   0.88      0.80      0.82     39600

#### Key Insights:
- The performance dropped for label 3, had a very poor precision of 0.16.
- Had a sub par performance for label 1.
- Had a good performance for the label 0.

# Logistic Regression Hyperparameter tuning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

lr = LogisticRegression(
    max_iter=2000,
    n_jobs=-1
)

param_grid = {
    'C':[0.01,0.1,1,5,10]
    }

lr_regressor = GridSearchCV(
    lr,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2)

lr_regressor.fit(X_train_final,y_train)

In [ ]:
print(lr_regressor.best_params_)

#### Best parameters:
- C = 10

# Logistic Regression final

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    solver='saga',
    C=10,
    max_iter=2000,
    tol=1e-3,
    random_state=42
)

In [ ]:
lr_model.fit(X_train_final, y_train)

In [ ]:
y_pred = lr_model.predict(X_val_final)

In [ ]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95     22835
           1       0.70      0.66      0.68      3183
           2       0.64      0.66      0.65     12488
           3       0.62      0.58      0.60      1094

    accuracy                           0.83     39600
    macro avg      0.73      0.72      0.72     39600
    weighted avg   0.83      0.83      0.83     39600

#### # Key Insights:
- Tuned hyperparametetrs improved performance for label 1 and label 3 to some degree.
- Also faced a slight drop in precision for label 1.

# SVM

In [ ]:
train = train_backup.copy()

from sklearn.model_selection import train_test_split

y = train['label']
x = train.drop('label',axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    x,y,
    train_size=0.8,
    random_state=42,
    stratify=y
)

### Transformers

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

word_vectorizer = TfidfVectorizer(
    stop_words='english',
    min_df=5,
    max_df=0.9,
    max_features=50000,
    ngram_range=(1,2)
)

char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    min_df=3,
    max_features=40000
)

ohe = OneHotEncoder(handle_unknown='ignore')

std_scaler = StandardScaler(with_mean = False)

### Pipeline Definition

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

text_col = ['comment']

cat_cols = ['race', 'religion', 'gender']

num_cols = [
    'post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', 'if_1',
    'if_2', 'disability', 'vote_score', 'vote_total', 'month', 'year', 'day', 'week_day',
    'is_weekend', 'total_emoticons', 'comment_length', 'word_count'
]

X_train['comment'] = X_train['comment'].str.lower()
X_val['comment'] = X_val['comment'].str.lower()

preprocessing = ColumnTransformer([
    ('word_tfidf', word_vectorizer, text_col),
    ('char_tfidf', char_vectorizer, text_col),
    ('OneHotEncoding', ohe, cat_cols),
    ('Scaling', std_scaler, num_cols)
])

### Training

In [ ]:
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('svm', LinearSVC(
        class_weight='balanced',
        max_iter=5000
        )
    )
])

svm_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_pred = svm_pipeline.predict(X_val)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.91      0.77     22835
           1       0.56      0.81      0.66      3183
           2       0.73      0.13      0.21     12488
           3       0.43      0.66      0.52      1094

    accuracy                           0.65     39600
    macro avg      0.60      0.63      0.54     39600
    weighted avg   0.67      0.65      0.58     39600

#### Key Insights:
- The model had a worse performance than logistic regression in all categories.
- Rather than hyperparameter tuning it, switched to next model. 

# LightGBM

In [ ]:
train = train_backup.copy()

### Train - Val Split

In [ ]:
from sklearn.model_selection import train_test_split

y = train['label']
X = train.drop('label', axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape)
print(X_val.shape)

### Categorical data handling

In [ ]:
cat_cols = ['race', 'religion', 'gender']

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_val[col] = X_val[col].cat.set_categories(X_train[col].cat.categories)

X_train['disability'] = X_train['disability'].astype('int')
X_val['disability'] = X_val['disability'].astype('int')

### Text Feature handling

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# WORD TFIDF
word_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=30000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

# CHAR TFIDF
char_tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    max_features=20000,
    min_df=3
)

In [ ]:
X_train_word = word_tfidf.fit_transform(X_train['comment'])
X_val_word = word_tfidf.transform(X_val['comment'])

X_train_char = char_tfidf.fit_transform(X_train['comment'])
X_val_char = char_tfidf.transform(X_val['comment'])

In [ ]:
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack

# COMBINE
X_train_combined = hstack([X_train_word, X_train_char])
X_val_combined = hstack([X_val_word, X_val_char])

# SVD
svd = TruncatedSVD(n_components=500, random_state=42)

X_train_text = svd.fit_transform(X_train_combined)
X_val_text = svd.transform(X_val_combined)

X_train_text.shape

In [ ]:
X_train = X_train.drop(columns=['comment'])
X_val = X_val.drop(columns=['comment'])

## Combining Features

In [ ]:
svd_cols = [f'svd_{i}' for i in range(X_train_text.shape[1])]

X_train_text_df = pd.DataFrame(X_train_text, columns=svd_cols, index=X_train.index)
X_val_text_df = pd.DataFrame(X_val_text, columns=svd_cols, index=X_val.index)

In [ ]:
X_train_final = pd.concat([X_train, X_train_text_df], axis=1)
X_val_final = pd.concat([X_val, X_val_text_df], axis=1)

## Training

In [ ]:
import lightgbm as lgb

cat_features = ['race', 'religion', 'gender']

train_dataset = lgb.Dataset(
    X_train_final,
    label=y_train,
    categorical_feature=cat_features,
    free_raw_data=False
)

val_dataset = lgb.Dataset(
    X_val_final,
    label=y_val,
    categorical_feature=cat_features,
    reference=train_dataset,
    free_raw_data=False
)

In [ ]:
params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'verbosity': -1
}

model = lgb.train(
    params,
    train_dataset,
    valid_sets=[train_dataset, val_dataset],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(100)]
)

In [ ]:
preds = model.predict(X_val_final)
y_pred = preds.argmax(axis=1)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96     22835
           1       0.76      0.73      0.74      3183
           2       0.82      0.90      0.86     12488
           3       0.77      0.30      0.44      1094

    accuracy                           0.90     39600
    macro avg      0.83      0.72      0.75     39600
    weighted avg   0.90      0.90      0.90     39600

#### Key Insights:
- The model had a great performance overall but had a very poor recall for label 3.
- LightGBM had a really good performance even at baseline.

# LightGBM with Stratified K fold with SVD

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = train.drop('label', axis=1)
y = train['label']

cat_cols = ['race', 'religion', 'gender']

params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'verbosity': -1,

    # GPU
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 63
}

word_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=30000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

char_tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,5),
    max_features=20000,
    min_df=3
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack
import lightgbm as lgb


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X), 4))


for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nFOLD {fold+1} ")

    X_train = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_train = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    for col in cat_cols:
        X_train[col] = X_train[col].astype('category')
        X_val[col] = X_val[col].astype('category')
        X_val[col] = X_val[col].cat.set_categories(X_train[col].cat.categories)

    X_train['disability'] = X_train['disability'].astype('int')
    X_val['disability'] = X_val['disability'].astype('int')

    
    X_train_word = word_tfidf.fit_transform(X_train['comment'])
    X_val_word = word_tfidf.transform(X_val['comment'])

    X_train_char = char_tfidf.fit_transform(X_train['comment'])
    X_val_char = char_tfidf.transform(X_val['comment'])

    
    X_train_combined = hstack([X_train_word, X_train_char])
    X_val_combined = hstack([X_val_word, X_val_char])

    
    svd = TruncatedSVD(n_components=500, random_state=42)
    X_train_text = svd.fit_transform(X_train_combined)
    X_val_text = svd.transform(X_val_combined)

    
    X_train = X_train.drop(columns=['comment'])
    X_val = X_val.drop(columns=['comment'])

    
    svd_cols = [f'svd_{i}' for i in range(X_train_text.shape[1])]

    X_train_text_df = pd.DataFrame(X_train_text, columns=svd_cols, index=X_train.index)
    X_val_text_df = pd.DataFrame(X_val_text, columns=svd_cols, index=X_val.index)

    X_train_final = pd.concat([X_train, X_train_text_df], axis=1)
    X_val_final = pd.concat([X_val, X_val_text_df], axis=1)

    
    train_dataset = lgb.Dataset(
        X_train_final,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_dataset = lgb.Dataset(
        X_val_final,
        label=y_val,
        categorical_feature=cat_cols,
        reference=train_dataset,
        free_raw_data=False
    )

    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[val_dataset],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(100)]
    )

    oof_preds[val_idx] = model.predict(X_val_final)
    print('Fold {fold + 1} ended')

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y, oof_preds.argmax(axis=1)))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96    114173
           1       0.75      0.76      0.75     15918
           2       0.85      0.86      0.85     62440
           3       0.54      0.54      0.54      5469

    accuracy                           0.90    198000
    macro avg      0.77      0.78      0.78    198000
    weighted avg   0.90      0.90      0.90    198000

#### Key Insights:
- The model had a great performance for label 0, 1 and 2.
- It's subpar performance in label 3 held it back.
- Stratified K fold helped the model by reducing the effect of data variance.

# LightGBM with Stratified K Fold NO SVD

In [ ]:
train = train_backup.copy() 

### Label Encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['race', 'religion', 'gender']

X = train.drop('label', axis=1)
y = train['label']
X_test = test.copy()

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

In [ ]:
tab_cols = []
for c in X.columns:
    if c != 'comment':
        tab_cols.append(c)

### Training

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'class_weight': 'balanced',
    'verbosity': -1,

    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 63
}

word_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    max_features=5000,
    min_df=5,
    sublinear_tf=True
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix
from scipy.optimize import minimize
import lightgbm as lgb
import gc

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X), 4))
test_preds = np.zeros((len(X_test), 4))


for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n FOLD {fold+1} ")

    X_train = X.iloc[tr_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    
    print('TF-IDF')
    X_train_word = word_tfidf.fit_transform(X_train['comment'])
    X_val_word = word_tfidf.transform(X_val['comment'])
    X_test_word = word_tfidf.transform(X_test['comment'])

    X_train_char = char_tfidf.fit_transform(X_train['comment'])
    X_val_char = char_tfidf.transform(X_val['comment'])
    X_test_char = char_tfidf.transform(X_test['comment'])


    print('Combining')
    X_train_tab_sp = csr_matrix(X_train[tab_cols].values.astype('float32'))
    X_val_tab_sp = csr_matrix(X_val[tab_cols].values.astype('float32'))
    X_test_tab_sp = csr_matrix(X_test[tab_cols].values.astype('float32'))

    
    X_train_final = hstack([X_train_tab_sp, X_train_word, X_train_char], format='csr')
    X_val_final = hstack([X_val_tab_sp, X_val_word, X_val_char], format='csr')
    X_test_final = hstack([X_test_tab_sp, X_test_word, X_test_char], format='csr')

    
    del X_train_word, X_val_word, X_test_word
    del X_train_char, X_val_char, X_test_char
    del X_train_tab_sp, X_val_tab_sp, X_test_tab_sp
    gc.collect()

    
    print('Training')
    train_dataset = lgb.Dataset(X_train_final, label=y_train, free_raw_data=True)
    val_dataset = lgb.Dataset(X_val_final, label=y_val, reference=train_dataset, free_raw_data=True)

    
    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[val_dataset],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(100)]
    )

    
    oof_preds[val_idx] = model.predict(X_val_final)
    test_preds += model.predict(X_test_final) / 5

    
    del X_train_final, X_val_final, X_test_final
    del train_dataset, val_dataset, model
    gc.collect()

    
    print(f"Fold {fold+1} done.\n")

In [ ]:
print("Macro F1:", f1_score(y, oof_preds.argmax(axis=1), average='macro'))
print(classification_report(y, oof_preds.argmax(axis=1)))

V1 Results:

Macro F1: 0.8067892317076966

           precision    recall  f1-score   support

           0       0.98      0.95      0.96    114173
           1       0.80      0.78      0.79     15918
           2       0.85      0.92      0.88     62440
           3       0.77      0.48      0.59      5469

    accuracy                           0.91    198000
    macro avg      0.85      0.78      0.81    198000
    weighted avg   0.92      0.91      0.91    198000

#### Key Insights:
- The model had a great overall performance and cleared the cutoff.
- The subpar recall in label 3 seems to be the deterent in this as well.
- Removing SVD likely allowed the model to focus more on the most important text feature.
- An importance focused model yield better results than more diverse features.

### HYPERPARAMETER TUNING

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)
char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=5000,
    min_df=5,
    sublinear_tf=True
)

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_data = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):

    X_tr_word  = word_tfidf.fit_transform(X.iloc[tr_idx]['comment'])
    X_val_word = word_tfidf.transform(X.iloc[val_idx]['comment'])

    X_tr_char  = char_tfidf.fit_transform(X.iloc[tr_idx]['comment'])
    X_val_char = char_tfidf.transform(X.iloc[val_idx]['comment'])

    X_tr_tab  = csr_matrix(X.iloc[tr_idx][tab_cols].values.astype('float32'))
    X_val_tab = csr_matrix(X.iloc[val_idx][tab_cols].values.astype('float32'))

    X_tr_final  = hstack([X_tr_tab, X_tr_word, X_tr_char], format='csr')
    X_val_final = hstack([X_val_tab, X_val_word, X_val_char], format='csr')

    fold_data.append((tr_idx, val_idx, X_tr_final, X_val_final))

    del X_tr_word, X_val_word, X_tr_char, X_val_char, X_tr_tab, X_val_tab
    gc.collect()

In [ ]:
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
import gc
import random

N_TRIALS = 15 # Itne random sets ke liye test hoga

search_space = {
    'num_leaves':        [64, 100, 200],
    'min_child_samples': [10, 20, 30],
    'feature_fraction':  [0.6, 0.8],
    'reg_alpha':         [0, 0.1, 0.5],
    'reg_lambda':        [0, 0.5, 1.0],
}

base_params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'class_weight': 'balanced',
    'verbosity': -1,
    'seed': 42,

    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 63,
}


random.seed(42)
def sample_params():
    return {k: random.choice(v) for k, v in search_space.items()}


best_score = -1
best_params_ = None
results = []


for i in range(N_TRIALS):

    print(f'TRIAL {i}')
    sampled = sample_params()
    trial_params = {**base_params, **sampled}
    oof_preds = np.zeros((len(X), 4))

    print('Creating Dataset')
    for tr_idx, val_idx, X_tr, X_val in fold_data:
        dtrain = lgb.Dataset(X_tr, label=y.iloc[tr_idx], free_raw_data=False)
        dval   = lgb.Dataset(X_val, label=y.iloc[val_idx], reference=dtrain, free_raw_data=False)

        model = lgb.train(
            trial_params, dtrain, valid_sets=[dval],
            num_boost_round=2000,
            callbacks=[lgb.early_stopping(100)],
        )
        oof_preds[val_idx] = model.predict(X_val)
        del dtrain, dval, model
        gc.collect()

    score = f1_score(y, oof_preds.argmax(axis=1), average='macro')
    results.append({**sampled, 'macro_f1': score})

    if score > best_score:
        best_score = score
        best_params_ = sampled


print("BEST PARAMS:")
print(best_params_)
print(f"Macro F1: {best_score}")

BEST PARAMS:

{'num_leaves':200,'min_child_samples':20,'feature_fraction':0.6,'reg_alpha':0.1,'reg_lambda':0.5}

Macro F1: 0.8101131251567261

### FINAL MODEL
- Tuned parameters
- Slower learning rate (0.03) for more precise learning.
- Higher num_boost_round	(3000) for longer training.
- Higher word TF-IDF max_features	(20000)
- Higher char TF-IDF max_features	(8000)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=8000,
    min_df=5,
    sublinear_tf=True
)

params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.03,
    'num_leaves': 200,
    'min_child_samples': 20,
    'feature_fraction': 0.6,
    'reg_alpha': 0.1,
    'reg_lambda': 0.5,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,

    'class_weight': 'balanced',
    'verbosity': -1,
    'seed': 42,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 63
}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix
from scipy.optimize import minimize
import lightgbm as lgb
import gc

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X), 4))
test_preds = np.zeros((len(X_test), 4))


for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n FOLD {fold+1} ")

    X_train = X.iloc[tr_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]


    print('TF-IDF')
    X_train_word = word_tfidf.fit_transform(X_train['comment'])
    X_val_word = word_tfidf.transform(X_val['comment'])
    X_test_word = word_tfidf.transform(X_test['comment'])

    X_train_char = char_tfidf.fit_transform(X_train['comment'])
    X_val_char = char_tfidf.transform(X_val['comment'])
    X_test_char = char_tfidf.transform(X_test['comment'])


    print('Combining')
    X_train_tab_sp = csr_matrix(X_train[tab_cols].values.astype('float32'))
    X_val_tab_sp = csr_matrix(X_val[tab_cols].values.astype('float32'))
    X_test_tab_sp = csr_matrix(X_test[tab_cols].values.astype('float32'))

    X_train_final = hstack([X_train_tab_sp, X_train_word, X_train_char], format='csr')
    X_val_final = hstack([X_val_tab_sp, X_val_word, X_val_char], format='csr')
    X_test_final = hstack([X_test_tab_sp, X_test_word, X_test_char], format='csr')

    del X_train_word, X_val_word, X_test_word
    del X_train_char, X_val_char, X_test_char
    del X_train_tab_sp, X_val_tab_sp, X_test_tab_sp
    gc.collect()
    
    
    print('Training')
    train_dataset = lgb.Dataset(X_train_final, label=y_train, free_raw_data=True)
    val_dataset = lgb.Dataset(X_val_final, label=y_val, reference=train_dataset, free_raw_data=True)

    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[val_dataset],
        num_boost_round=3000,
        callbacks=[lgb.early_stopping(100)]
    )

    oof_preds[val_idx] = model.predict(X_val_final)
    test_preds += model.predict(X_test_final) / 5

    del X_train_final, X_val_final, X_test_final
    del train_dataset, val_dataset, model
    gc.collect()

    print(f"Fold {fold+1} done.\n")

In [ ]:
print("Macro F1:", f1_score(y, oof_preds.argmax(axis=1), average='macro'))
print(classification_report(y, oof_preds.argmax(axis=1)))

Macro F1: 0.8104165205943268

               precision    recall  f1-score   support

           0       0.98      0.95      0.96    114173
           1       0.80      0.78      0.79     15918
           2       0.85      0.92      0.88     62440
           3       0.77      0.50      0.60      5469

      accuracy                         0.91    198000
      macro avg    0.85      0.79      0.81    198000
      weighted avg 0.92      0.91      0.91    198000

#### Key Insights:
- The slight increase in recall of label 3 improved the models performance. 
- Increasing the number of max features and tuning the HP improved the model performance.

### Threshold Tuning

In [ ]:
y_true = y.values

def macro_f1_threshold(thresholds):
    adjusted = oof_preds.copy()
    for i in range(4):
        adjusted[:, i] /= thresholds[i]
    return -f1_score(y_true, adjusted.argmax(axis=1), average='macro')

result = minimize(macro_f1_threshold, [1, 1, 1, 1], method='Nelder-Mead')
best_thresholds = result.x
print("Best thresholds:", best_thresholds)

adjusted = oof_preds.copy()
for i in range(4):
    adjusted[:, i] /= best_thresholds[i]

print("\n Threshold tuned results")
print("Macro F1:", f1_score(y_true, adjusted.argmax(axis=1), average='macro'))
print(classification_report(y_true, adjusted.argmax(axis=1)))

Macro F1: 0.8241332389378435

               precision    recall  f1-score   support

           0       0.97      0.95      0.96    114173
           1       0.79      0.80      0.80     15918
           2       0.87      0.90      0.89     62440
           3       0.64      0.68      0.66      5469

      accuracy                          0.91    198000
      macro avg     0.82      0.83      0.82    198000
      weighted avg  0.92      0.91      0.92    198000

# MODEL PERFORMANCE COMPARISION

In [ ]:
model_performance = {"Logistic Regression Baseline": 0.64,
                     "Logistic Regression Final": 0.72,
                     "SVM": 0.54,
                     "LightGBM Baseline": 0.75,
                     "LightGBM with Stratified K Fold and SVD": 0.78,
                     "LightGBM with Stratified K Fold": 0.806,
                     "HP Tuned LightGBM with Stratified K Fold": 0.810,
                     "Threshold Tuned LightGBM with Stratified K Fold":0.824,
                    }
model = model_performance.keys()
score = model_performance.values()

plt.figure(figsize=(10,6))
plt.barh(model, score)
plt.xlabel("Score")
plt.title("Model Performance Comparison")

# TESTING

### TESTING LIGHTGBM NO SVD

In [ ]:
test_adjusted = test_preds.copy()
for i in range(4):
    test_adjusted[:, i] /= best_thresholds[i]

test_labels = test_adjusted.argmax(axis=1)

In [ ]:
submission = pd.DataFrame({
    'ID': range(1, len(test_labels) + 1),
    'label': test_labels
})

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission.head(15))
print(f"\nSubmission shape: {submission.shape}")
print(f"Label distribution:\n{submission['label'].value_counts().sort_index()}")

In [ ]:
s = pd.read_csv('submission.csv')
s.head()

# APPENDIX

### MILESTONE 1

In [ ]:
'''# For training shape, no. of columns in test
print('Shape of training data: ', train.shape)
print('Shape of testing data: ', test.shape)'''

In [ ]:
'''# For object, numerical and boolean datatypes
train.dtypes'''

In [ ]:
'''# For missing values
train.isnull().sum()'''

In [ ]:
'''# For distinct labels
train['label'].nunique()'''

In [ ]:
'''# For distribution share of 0
train['label'].value_counts(normalize=True)'''

In [ ]:
'''# For median of upvotes
train.describe()'''

### MILESTONE 2

In [ ]:
'''# Most common month
train['created_date'] = pd.to_datetime(train['created_date'])

train['month'] = train['created_date'].dt.month_name()
most_common_month = train['month'].value_counts().idxmax()
most_common_month'''

In [ ]:
'''# Get max value of total emopticon 
train['total_emoticons'] = (train['emoticon_1'] + train['emoticon_2'] + train['emoticon_3'])
max_value = train['total_emoticons'].max()
max_value'''

In [ ]:
'''# Median character length of comment where label = 3
train3 = train[train['label'] == 3].copy()
train3['comment'] = train3['comment'].fillna('')
train3['com_len'] = train3['comment'].str.len()
train3.describe()'''

In [ ]:
'''# Avg word count of label 1 comments
train1 = train[train['label'] == 1].copy()
train1['words'] = train1['comment'].fillna('').str.split()
train1['w_count'] = train1['words'].str.len()
train1.describe()'''

In [ ]:
'''# Trump count in comments
count = (train['comment'].fillna('').str.contains('Trump', case=False).sum())
int(count)'''

In [ ]:
'''# Number of non stop words
stop_words = [
    'a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to', 
    'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 
    'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his', 
    'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your'
    ]

text = str(train.loc[0, 'comment']).lower()
# print(text)
s = ''
for i in text:
    if i not in ['.', ',', '!', '?', ':', ';', '"', "'"]:
        s += i

words = s.split()
clean_words = []
for word in words:
    if word not in stop_words:
        clean_words.append(word)

print(len(clean_words))'''

In [ ]:
'''# Unique words
unique_tokens = set()

for text in train['comment'].fillna(''):
    text = text.lower()
    words = text.split()
    
    for word in words:
        unique_tokens.add(word)

print(len(unique_tokens))'''

In [ ]:
'''# Tfidf feature generation
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    min_df=5,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(train['comment'].fillna(''))

num_features = len(vectorizer.get_feature_names_out())
print(num_features)'''

### MILESTONE 3

In [ ]:
'''# Split the train dataset
from sklearn.model_selection import train_test_split

y = train['label']
x = train.drop('label',axis=1)

X_train, X_val, y_train, y_val = train_test_split(
    x,y,
    train_size=0.8,
    random_state=42,
    stratify=y
)

a,b = X_train.shape
c,d = X_val.shape
a+c'''

In [ ]:
'''X_train['created_date'] = pd.to_datetime(X_train['created_date'],errors = 'coerce')
X_train['month']= X_train['created_date'].dt.month
X_train['year'] = X_train['created_date'].dt.year
X_train['day'] = X_train['created_date'].dt.day

X_val['created_date'] = pd.to_datetime(X_val['created_date'],errors = 'coerce')
X_val['month']= X_val['created_date'].dt.month
X_val['year'] = X_val['created_date'].dt.year
X_val['day'] = X_val['created_date'].dt.day

print(X_train['month'].mode())'''

In [ ]:
'''# One hot encoding
to_encode_columns = ['religion', 'gender', 'race']
X_train[to_encode_columns] = X_train[to_encode_columns].fillna('none')
X_val[to_encode_columns] = X_val[to_encode_columns].fillna('none')

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

ohe.fit(X_train[to_encode_columns])

encoded_col_train = ohe.transform(X_train[to_encode_columns])
encoded_col_train = pd.DataFrame(encoded_col_train,
    columns=ohe.get_feature_names_out(to_encode_columns),
    index=X_train.index
)

encoded_col_val = ohe.transform(X_val[to_encode_columns])
encoded_col_val = pd.DataFrame(encoded_col_val,
    columns=ohe.get_feature_names_out(to_encode_columns),
    index=X_val.index
)

X_train = X_train.drop(to_encode_columns, axis=1)
X_val = X_val.drop(to_encode_columns, axis=1)

X_train = pd.concat([X_train, encoded_col_train], axis=1)
X_val = pd.concat([X_val, encoded_col_val], axis=1)

print(X_train.shape)'''

In [ ]:
'''# CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_comments = X_train['comment'].fillna('')
X_val_comments = X_val['comment'].fillna('')

vectorizer.fit(X_train_comments)

X_train_spmat = vectorizer.transform(X_train_comments)
X_val_spmat = vectorizer.transform(X_val_comments)

row_sum = X_train_spmat[1].sum()
print(row_sum)'''

In [ ]:
'''# Disability feature
X_train['disability'] = X_train['disability'].astype(int)
X_val['disability'] = X_val['disability'].astype(int)

print(X_train['disability'].sum() + X_val['disability'].sum())'''

In [ ]:
'''# Standard Scaler
X_train_stdsc = X_train.drop(columns = 'created_date',axis = 1)
X_val_stdsc = X_val.drop(columns = 'created_date',axis = 1)
numeric_cols = X_train_stdsc.select_dtypes(include=['number']).columns
print(len(numeric_cols))

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train_stdsc[numeric_cols])'''

In [ ]:
'''# Splitting train test

from sklearn.model_selection import train_test_split

X = train.drop('label', axis=1)
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)'''

In [ ]:
'''# Imputing
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')

imputer.fit(X_train)

X_train = pd.DataFrame(imputer.transform(X_train), columns=X_train.columns)
X_val = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns)'''

In [ ]:
'''# Removing negatives
numeric_cols = ['post_id', 'emoticon_1', 'emoticon_2', 'emoticon_3','upvote', 'downvote', 'if_1', 'if_2']

for col in numeric_cols:
    X_train[col] = X_train[col].astype(float)
    X_val[col] = X_val[col].astype(float)

X_train[numeric_cols] = X_train[numeric_cols].abs()
X_val[numeric_cols] = X_val[numeric_cols].abs()'''

In [ ]:
'''# Creating date features
X_train['created_date'] = pd.to_datetime(X_train['created_date'],errors = 'coerce')
X_train['month']= X_train['created_date'].dt.month
X_train['year'] = X_train['created_date'].dt.year
X_train['day'] = X_train['created_date'].dt.day

X_val['created_date'] = pd.to_datetime(X_val['created_date'],errors = 'coerce')
X_val['month']= X_val['created_date'].dt.month
X_val['year'] = X_val['created_date'].dt.year
X_val['day'] = X_val['created_date'].dt.day

X_train = X_train.drop('created_date', axis=1)
X_val = X_val.drop('created_date', axis=1)'''

In [ ]:
'''# Extraing comment features
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(stop_words='english')
tf.fit(X_train['comment'])

X_train_vec = tf.transform(X_train['comment'])
X_val_vec = tf.transform(X_val['comment'])

X_train = X_train.drop('comment', axis=1)
X_val = X_val.drop('comment', axis=1)'''

In [ ]:
'''# One hot encoding
from sklearn.preprocessing import OneHotEncoder
cat_cols = ['race', 'religion', 'gender']
ohe = OneHotEncoder(handle_unknown='ignore')

ohe.fit(X_train[cat_cols])

X_train_cat = ohe.transform(X_train[cat_cols])
X_val_cat = ohe.transform(X_val[cat_cols])

X_train = X_train.drop(cat_cols, axis=1)
X_val = X_val.drop(cat_cols, axis=1)'''

### MILESTONE 4

In [ ]:
'''# Q1
from sklearn.model_selection import train_test_split

train['comment'] = train['comment'].fillna('')

X = train.drop(columns=['label'])
y = train['label']

X_train_A, X_val_A, y_train_A, y_val_A = train_test_split(
    X, y, test_size=0.4, random_state=2306, stratify=y
)

train_counts_A = np.array(y_train_A.value_counts().sort_index())
val_counts_A   = np.array(y_val_A.value_counts().sort_index())

print(train_counts_A)
print(val_counts_A)

X_train_B, X_val_B, y_train_B, y_val_B = train_test_split(
    X, y, test_size=0.4, random_state=2306, stratify=None
)

train_counts_B = np.array(y_train_B.value_counts().sort_index())
val_counts_B   = np.array(y_val_B.value_counts().sort_index())

print(train_counts_B)
print(val_counts_B)

prop_A = val_counts_A / val_counts_A.sum()
prop_B = val_counts_B / val_counts_B.sum()

max_diff = np.max(np.abs(prop_A - prop_B))
max_diff'''

In [ ]:
'''# Q2
X_train = X_train_A.copy().drop(columns=['created_date'])
X_test  = X_val_A.copy().drop(columns=['created_date'])
y_train = y_train_A.copy()
y_test  = y_val_A.copy()

text_X_train = X_train['comment']
text_X_test  = X_test['comment']
X_train = X_train.drop(columns=['comment'])
X_test  = X_test.drop(columns=['comment'])

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

cat_cols = ["race", "religion", "gender", "disability"]
num_cols = ["post_id", "emoticon_1", "emoticon_2", "emoticon_3",
            "upvote", "downvote", "if_1", "if_2"]

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ],
    remainder='passthrough'
)

X_train_tabular = preprocessor.fit_transform(X_train)
X_test_tabular  = preprocessor.transform(X_test)

import re
def normalize_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

text_X_train_norm = text_X_train.apply(normalize_text)
text_X_test_norm  = text_X_test.apply(normalize_text)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tf_idf_train = tfidf.fit_transform(text_X_train_norm)
tf_idf_test  = tfidf.transform(text_X_test_norm)

from scipy.sparse import hstack
X_train_final = hstack([X_train_tabular, tf_idf_train])
X_test_final  = hstack([X_test_tabular, tf_idf_test])

print(X_train_final.shape)
print(X_test_final.shape)
print(X_train_final.sum())'''

In [ ]:
'''# Q3
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

svd = TruncatedSVD(n_components=300, random_state=2306)
X_train_reduced = svd.fit_transform(X_train_final)
X_test_reduced  = svd.transform(X_test_final)

params = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [5, 10, 15]
}

rf = RandomForestClassifier(random_state=2306)

randomized_search = RandomizedSearchCV(
    rf,
    param_distributions=params,
    n_iter=5,
    cv=3,
    random_state=2306,
    n_jobs=-1
)

randomized_search.fit(X_train_reduced, y_train)

print(randomized_search.best_params_)'''

In [ ]:
'''# Q4
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=50, random_state=2306)
ada.fit(X_train_reduced, y_train)

ada_est_error = ada.estimator_errors_
variance = np.var(ada_est_error)
variance'''

In [ ]:
'''# Q5
from sklearn.ensemble import RandomForestClassifier

rf5 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2306)
rf5.fit(X_train_reduced, y_train)

rf_feature_importance = rf5.feature_importances_ 
max_idx = np.argmax(rf_feature_importance)

print(rf5.feature_importances_[max_idx])
max_idx'''

In [ ]:
'''# Q6
N = X_train_reduced.shape[1]

input_layer = N
h1 = 128
h2 = 64
h3 = 32
output_layer = 4

total_weights = (input_layer * h1 + h1 * h2 + h2 * h3 + h3 * output_layer)

print(total_weights)'''

In [ ]:
'''# Q7
from sklearn.neural_network import MLPClassifier

mlp_clf = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=5,
    batch_size=32,
    random_state=2306
)

mlp_clf.fit(X_train_reduced, y_train)

mlp_clf.loss_'''

In [ ]:
'''# Q8
from sklearn.metrics import f1_score

mlp_A = MLPClassifier(hidden_layer_sizes=(100,), max_iter=5, random_state=2306, alpha=0.0001)
mlp_B = MLPClassifier(hidden_layer_sizes=(100,), max_iter=5, random_state=2306, alpha=1.0)

mlp_A.fit(X_train_reduced, y_train)
mlp_B.fit(X_train_reduced, y_train)

f1_A = f1_score(y_train, mlp_A.predict(X_train_reduced), average='macro')
f1_B = f1_score(y_train, mlp_B.predict(X_train_reduced), average='macro')

print(f1_A)
print(f1_B)
print(abs(f1_A - f1_B))'''

In [ ]:
'''# Q9
from sklearn.metrics import confusion_matrix

y_pred_val = mlp_clf.predict(X_test_reduced)
cm = confusion_matrix(y_test, y_pred_val)

off_diagonal_sum = cm.sum() - np.trace(cm)
misclassification_rate = off_diagonal_sum / len(y_test)

print('Confusion Matrix:')
print(cm)

print(misclassification_rate)'''

### MILESTONE 5

In [ ]:
''' # Q1, Q2, Q3

df = train.copy()

y = df['label']
df = df.drop('label', axis=1)

df['created_date'] = pd.to_datetime(df['created_date'])
test_df['created_date'] = pd.to_datetime(test_df['created_date'])

for x in [df,test_df]:
  x['year'] = x['created_date'].dt.year
  x['month']= x['created_date'].dt.month
  x['hour'] = x['created_date'].dt.hour
df = df.drop('created_date',axis=1)
test_df = test_df.drop('created_date',axis=1)

cat_cols = ['race', 'religion', 'gender']
for x in [df, test_df]:
    x[cat_cols] = x[cat_cols].fillna("Unknown")
    x['comment'] = x['comment'].fillna("")
    x['disability'] = x['disability'].astype(int)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english', max_features=5000), 'comment'),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['race','religion','gender'])
    ],
    remainder='passthrough'
)

X = preprocessor.fit_transform(df)
X_test = preprocessor.transform(test_df)

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)
print(len(y_train))
print(len(y_val)) '''

In [ ]:
'''# Q4, Q5
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train, y_train)

y_pred = nb.predict(X_val)

from sklearn.metrics import accuracy_score, classification_report

print(accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))'''

In [ ]:
'''# Q6, Q7, Q8
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=500,
    random_state=42
)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_val)
y_train_pred = lr.predict(X_train)

from sklearn.metrics import accuracy_score, classification_report

print('Validation Accuracy', accuracy_score(y_val, y_pred_lr))
print('Training Accuracy', accuracy_score(y_train, y_train_pred))
print(classification_report(y_val, y_pred_lr))'''

In [ ]:
'''# Q9, Q10

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

lr_hpt = LogisticRegression(
    solver='liblinear',
    random_state=42,
    max_iter=500
)

param_grid = {
    'C': [0.1, 1, 10]
}

grid = GridSearchCV(
    lr_hpt,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)

best_lr = grid.best_estimator_

y_pred_grid = best_lr.predict(X_val)

from sklearn.metrics import accuracy_score, classification_report

print('Validation Accuracy', accuracy_score(y_val, y_pred_grid))'''

# Dummy Submission

In [ ]:
'''sample_sub['label'] = preds
sample_sub.to_csv('submission.csv',index=False)

from sklearn.dummy import DummyClassifier

x_train = train.drop(columns=['label'])
y_train = train['label']

dummy = DummyClassifier(strategy = 'most_frequent')
dummy.fit(x_train,y_train)

preds = dummy.predict(test)
preds'''